# myPolar – Deep Learning (Bi-LSTM, Transformer)

**Goal:** classify Burmese sentences for the 3 POLAR subtasks.

| Task | Question | Type |
|---|---|---|
| Task 1 | Is the sentence polarized? | binary (0 / 1) |
| Task 2 | Who is the target? (political, racial/ethnic, religious, gender/sexual, other) | multi-label |
| Task 3 | How is polarization shown? (stereotype, vilification, ...) | multi-label |

Same data and same tasks as `myPolar_ML.ipynb`, but now with neural networks.

| Model | Idea | Input text |
|---|---|---|
| **Bi-LSTM** | reads the sentence left→right and right→left | syllables (`text_syllable`) |
| **Transformer** (small, from scratch) | every syllable looks at every other syllable (self-attention) | syllables (`text_syllable`) |

**Plan:** train on *train* → keep the epoch with the best *val* score → choose thresholds on *val* → score once on *test*.

⏱️ Runs on a normal laptop (CPU or Apple M-chip GPU). Bi-LSTM and Transformer take about 10–20 minutes in total.

## Step 0. Install the packages (only the first time)
`%pip` installs into the same environment this notebook uses. After it finishes, restart the kernel once
(**Kernel → Restart Kernel**) and continue with Step 1.

In [5]:
%pip install -q torch scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


## Step 1. Settings and data

In [7]:
import os, random, copy, warnings
warnings.filterwarnings("ignore")                 # keep the output clean

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, classification_report

# ---------- settings you may change ----------
DATA_DIR   = "data"                # folder with the csv files
TEXT_COL   = "text_syllable"       # Bi-LSTM / Transformer read syllables
MAX_LEN    = 128                   # keep the first 128 syllables (95% of sentences are shorter)
BATCH_SIZE = 32
EPOCHS     = 10                    # passes over the training data
SEED       = 42

def set_seed(seed=SEED):
    """Same seed -> same random numbers -> same results every run."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

set_seed()

# use a GPU if we have one (cuda = NVIDIA, mps = Apple M1/M2/M3), otherwise the CPU
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("Device:", device)

# ---------- load train / dev / test for every task ----------
TASKS = {
    "task1": ("task1_polarization",  ["polarization"]),
    "task2": ("task2_type",          ["political", "racial_ethnic", "religious", "gender_sexual", "other"]),
    "task3": ("task3_manifestation", ["stereotype", "vilification", "dehumanization",
                                      "extreme_language", "lack_of_empathy", "invalidation"]),
}
data = {}
for task, (prefix, labels) in TASKS.items():
    data[task] = {split: pd.read_csv(f"{DATA_DIR}/{prefix}_{split}.csv", keep_default_na=False)
                  for split in ["train", "dev", "test"]}
    print(f"{task}: train={len(data[task]['train'])}  dev={len(data[task]['dev'])}  "
          f"test={len(data[task]['test'])}  labels={len(labels)}")

Device: mps
task1: train=7668  dev=1643  test=1644  labels=1
task2: train=3570  dev=766  test=771  labels=5
task3: train=3324  dev=710  test=709  labels=6


## Step 2. Vocabulary: syllable → number
A neural network needs numbers. We give every syllable an ID.
- `<pad>` (ID 0) fills short sentences up to `MAX_LEN`
- `<unk>` (ID 1) is used for syllables the model never saw in training

We build the vocabulary from **task1 train** only (task2/task3 train sentences are inside it), so dev/test stay unseen.

In [11]:
from collections import Counter

counter = Counter(tok for sentence in data["task1"]["train"][TEXT_COL] for tok in sentence.split())

vocab = {"<pad>": 0, "<unk>": 1}
for tok, count in counter.items():
    if count >= 2:                       # skip syllables seen only once (often typos)
        vocab[tok] = len(vocab)

def encode(sentence):
    """sentence -> list of MAX_LEN IDs (cut if too long, pad with 0 if too short)"""
    ids = [vocab.get(tok, 1) for tok in sentence.split()][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

example = data["task1"]["train"][TEXT_COL][0]
print("Vocabulary size:", len(vocab))
print("Syllables:", example.split()[:8])
print("IDs      :", encode(example)[:8])

Vocabulary size: 2115
Syllables: ['မ', 'ဟာ', 'သ', 'ရေ', 'စည်', 'သူ', 'ဦး', 'သ']
IDs      : [2, 3, 4, 5, 6, 7, 8, 4]


## Step 3. DataLoader: give the data to the model in small batches
Every sentence becomes a row of `MAX_LEN` syllable IDs. A batch = 32 sentences.

In [13]:
def make_loader(task, split, shuffle=False):
    df = data[task][split]
    labels = TASKS[task][1]
    X = torch.tensor([encode(s) for s in df[TEXT_COL]])              # sentences -> IDs
    y = torch.tensor(df[labels].values, dtype=torch.float)          # labels: 0 / 1
    return DataLoader(TensorDataset(X, y), batch_size=BATCH_SIZE, shuffle=shuffle)

X_batch, y_batch = next(iter(make_loader("task2", "train")))
print("One batch -> X:", tuple(X_batch.shape), " y:", tuple(y_batch.shape))

One batch -> X: (32, 128)  y: (32, 5)


## Step 4. The two models we build ourselves
Both models: **IDs → embedding vectors → encoder → average over the sentence → one score per label**.

The output has one number per label (Task 1: 1, Task 2: 5, Task 3: 6).
A **sigmoid** later turns each number into a probability between 0 and 1.

In [15]:
def average_pool(h, x):
    """Average the vectors of the real syllables (ignore <pad>)."""
    mask = (x != 0).unsqueeze(-1).float()                 # 1 = real syllable, 0 = padding
    return (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)


class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, num_labels, emb_dim=128, hidden=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)       # ID -> vector
        self.lstm = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)                                          # helps against overfitting
        self.out = nn.Linear(hidden * 2, num_labels)                            # *2 = forward + backward

    def forward(self, x):
        h, _ = self.lstm(self.embedding(x))       # (batch, MAX_LEN, 2*hidden)
        return self.out(self.dropout(average_pool(h, x)))


class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, num_labels, d_model=128, n_heads=4, n_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.position = nn.Embedding(MAX_LEN, d_model)                          # tells the model the word order
        layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=256,
                                           dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.dropout = nn.Dropout(0.3)
        self.out = nn.Linear(d_model, num_labels)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        h = self.embedding(x) + self.position(positions)
        h = self.encoder(h, src_key_padding_mask=(x == 0))  # do not attend to <pad>
        return self.out(self.dropout(average_pool(h, x)))


print("Bi-LSTM parameters    :", sum(p.numel() for p in BiLSTMClassifier(len(vocab), 1).parameters()))
print("Transformer parameters:", sum(p.numel() for p in TransformerClassifier(len(vocab), 1).parameters()))

Bi-LSTM parameters    : 535169
Transformer parameters: 552193


## Step 5. Helper functions: score, thresholds, training, testing
- **Score** = macro-F1 (same as the ML notebook).
- **Loss** = `BCEWithLogitsLoss` with `pos_weight`: mistakes on rare labels cost more.
- **Early stopping (simple)**: after every epoch we check dev and keep the best epoch.
- **Thresholds**: by default a label is "1" if probability ≥ 0.5. We try other cut-offs on **dev** and keep the best one for each label (never on test).

In [17]:
def macro_f1(task, y_true, y_pred):
    # Task 1: average F1 of class 0 and class 1.  Task 2/3: average F1 over the labels.
    if task == "task1":
        return f1_score(y_true.ravel(), y_pred.ravel(), average="macro", zero_division=0)
    return f1_score(y_true, y_pred, average="macro", zero_division=0)


def get_probs(model, loader):
    """Run the model without training and return probabilities and true labels."""
    model.eval()
    all_probs, all_y = [], []
    with torch.no_grad():
        for X, y in loader:
            logits = model(X.to(device))
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_y.append(y.numpy())
    return np.vstack(all_probs), np.vstack(all_y).astype(int)


def tune_thresholds(task, probs, y):
    """For each label, try cut-offs 0.05 ... 0.95 on dev and keep the best one."""
    grid = np.arange(0.05, 0.96, 0.05)
    best = []
    for j in range(y.shape[1]):
        scores = [macro_f1(task, y[:, [j]], (probs[:, [j]] >= t).astype(int)) if task == "task1"
                  else f1_score(y[:, j], (probs[:, j] >= t).astype(int), zero_division=0)
                  for t in grid]
        best.append(grid[int(np.argmax(scores))])
    return np.array(best)


def train_model(model, task, lr, epochs=EPOCHS):
    set_seed()
    model = model.to(device)
    train_loader = make_loader(task, "train", shuffle=True)
    dev_loader   = make_loader(task, "dev")

    # pos_weight = (#negatives / #positives) for every label -> rare labels get a bigger weight
    y_train = data[task]["train"][TASKS[task][1]].values
    pos_weight = torch.tensor((len(y_train) - y_train.sum(0)) / y_train.sum(0).clip(min=1),
                              dtype=torch.float).to(device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    best_f1, best_state = -1, None
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        for X, y in train_loader:
            optimizer.zero_grad()                                  # 1) clear old gradients
            logits = model(X.to(device))                           # 2) predict
            loss = loss_fn(logits, y.to(device))                   # 3) how wrong?
            loss.backward()                                        # 4) compute gradients
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)      #    (avoid too big steps)
            optimizer.step()                                       # 5) update the weights
            total_loss += loss.item()

        probs, y_dev = get_probs(model, dev_loader)
        dev_f1 = macro_f1(task, y_dev, (probs >= 0.5).astype(int))
        mark = ""
        if dev_f1 > best_f1:                                       # remember the best epoch
            best_f1, best_state, mark = dev_f1, copy.deepcopy(model.state_dict()), "  <- best"
        print(f"epoch {epoch:2d} | train loss {total_loss / len(train_loader):.4f} | dev macro-F1 {dev_f1:.4f}{mark}")

    model.load_state_dict(best_state)                              # go back to the best epoch
    return model


def evaluate(model, task, model_name):
    labels = TASKS[task][1]
    dev_probs,  y_dev  = get_probs(model, make_loader(task, "dev"))
    test_probs, y_test = get_probs(model, make_loader(task, "test"))

    thresholds = tune_thresholds(task, dev_probs, y_dev)            # chosen on dev only
    dev_f1  = macro_f1(task, y_dev,  (dev_probs  >= thresholds).astype(int))
    y_pred  = (test_probs >= thresholds).astype(int)
    test_f1 = macro_f1(task, y_test, y_pred)

    print(f"\n>>> {model_name} | {task} | thresholds {np.round(thresholds, 2).tolist()}")
    print(f">>> dev macro-F1 {dev_f1:.4f} | TEST macro-F1 {test_f1:.4f}")
    if task == "task1":
        print(classification_report(y_test.ravel(), y_pred.ravel(),
                                    target_names=["non-polarized", "polarized"], digits=3, zero_division=0))
    else:
        print(classification_report(y_test, y_pred, target_names=labels, digits=3, zero_division=0))
    return {"task": task, "model": model_name, "dev_macro_f1": round(dev_f1, 4), "test_macro_f1": round(test_f1, 4)}


results = []      # every final score is stored here

## Step 6. Bi-LSTM (one model per task)

In [19]:
for task in TASKS:
    print(f"\n========== Bi-LSTM | {task} ==========")
    model = BiLSTMClassifier(len(vocab), num_labels=len(TASKS[task][1]))
    model = train_model(model, task, lr=1e-3)
    results.append(evaluate(model, task, "BiLSTM"))


========== Bi-LSTM | task1 ==========
epoch  1 | train loss 0.6046 | dev macro-F1 0.7121  <- best
epoch  2 | train loss 0.5111 | dev macro-F1 0.7344  <- best
epoch  3 | train loss 0.4499 | dev macro-F1 0.7610  <- best
epoch  4 | train loss 0.3818 | dev macro-F1 0.7626  <- best
epoch  5 | train loss 0.3056 | dev macro-F1 0.7681  <- best
epoch  6 | train loss 0.2257 | dev macro-F1 0.7594
epoch  7 | train loss 0.1552 | dev macro-F1 0.7632
epoch  8 | train loss 0.0910 | dev macro-F1 0.7594
epoch  9 | train loss 0.0580 | dev macro-F1 0.7529
epoch 10 | train loss 0.0363 | dev macro-F1 0.7626

>>> BiLSTM | task1 | thresholds [0.45]
>>> dev macro-F1 0.7699 | TEST macro-F1 0.7682
               precision    recall  f1-score   support

non-polarized      0.772     0.765     0.769       827
    polarized      0.765     0.771     0.768       817

     accuracy                          0.768      1644
    macro avg      0.768     0.768     0.768      1644
 weighted avg      0.768     0.768     0.7

## Step 7. Transformer from scratch (one model per task)

In [20]:
for task in TASKS:
    print(f"\n========== Transformer | {task} ==========")
    model = TransformerClassifier(len(vocab), num_labels=len(TASKS[task][1]))
    model = train_model(model, task, lr=5e-4)
    results.append(evaluate(model, task, "Transformer"))


========== Transformer | task1 ==========
epoch  1 | train loss 0.6370 | dev macro-F1 0.6732  <- best
epoch  2 | train loss 0.5533 | dev macro-F1 0.6917  <- best
epoch  3 | train loss 0.4870 | dev macro-F1 0.7098  <- best
epoch  4 | train loss 0.4112 | dev macro-F1 0.7022
epoch  5 | train loss 0.3314 | dev macro-F1 0.7012
epoch  6 | train loss 0.2494 | dev macro-F1 0.7048
epoch  7 | train loss 0.1862 | dev macro-F1 0.6982
epoch  8 | train loss 0.1331 | dev macro-F1 0.7090
epoch  9 | train loss 0.1021 | dev macro-F1 0.7048
epoch 10 | train loss 0.0809 | dev macro-F1 0.7127  <- best

>>> Transformer | task1 | thresholds [0.6]
>>> dev macro-F1 0.7132 | TEST macro-F1 0.6959
               precision    recall  f1-score   support

non-polarized      0.701     0.689     0.695       827
    polarized      0.691     0.703     0.697       817

     accuracy                          0.696      1644
    macro avg      0.696     0.696     0.696      1644
 weighted avg      0.696     0.696     0.69

## Step 8. Compare all models
We also add the best traditional ML scores from `myPolar_ML.ipynb` (if `results/ml_test_results.csv` exists).

In [23]:
os.makedirs("results", exist_ok=True)
dl = pd.DataFrame(results)
dl.to_csv("results/dl_test_results.csv", index=False)

table = dl.copy()
ml_file = "results/ml_test_results.csv"
if os.path.exists(ml_file):
    ml = pd.read_csv(ml_file)
    ml["model"] = "Traditional ML (best)"                      # SGD/SVM + bigram TF-IDF
    table = pd.concat([ml[["task", "model", "dev_macro_f1", "test_macro_f1"]], table])

comparison = table.pivot_table(index="model", columns="task", values="test_macro_f1").round(3)
comparison.to_csv("results/all_test_results.csv")
print("TEST macro-F1 (higher is better):")
print(comparison)

TEST macro-F1 (higher is better):
task                   task1  task2  task3
model                                     
BiLSTM                 0.768  0.600  0.520
Traditional ML (best)  0.784  0.662  0.512
Transformer            0.696  0.531  0.518


## Results and findings

**Test macro-F1 (best epoch and thresholds chosen on dev):**

| Model | Task 1 | Task 2 | Task 3 |
|---|---|---|---|
| Bi-LSTM (syllable embeddings) | 0.768 | 0.600 | **0.520** |
| Transformer from scratch (2 layers, 4 heads) | 0.696 | 0.531 | 0.518 |
| Best traditional ML (for comparison) | **0.784** | **0.662** | 0.512 |

- With about 7,700 training sentences for Task 1 and about 3,500 for Tasks 2 and 3, neither network trained from scratch beat the linear baselines on Tasks 1 and 2.
- The Bi-LSTM was clearly stronger than the small Transformer, which is expected for small data.
- On Task 3 all three families are close (0.51–0.52), which suggests the label definitions and amount of data, not the model, are the main limit.